# 05. Feature Selection
##### DSR Berlin | Data Science Fundamentals
##### Inspired by Adam Green (adgefficiency.com), Rachel Berryman, and Samson Afolabi

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
1. Explain the **principle of parsimony (Occam's razor)** and why smaller, cleaner feature sets outperform bloated feature spaces.
2. Differentiate and implement the **three families of feature selection**:
   - **Filter Methods** (Variance Threshold, Correlation, Mutual Information)
   - **Wrapper Methods** (Recursive Feature Elimination - RFE)
   - **Embedded Methods** (Lasso L1 regularization, Tree-based Gini importance)
3. Evaluate feature importance using **Permutation Importance** to avoid cardinality bias.
4. Select the optimal subset of features for the Telco Churn dataset and persist them for modeling.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif, VarianceThreshold, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

import warnings

warnings.filterwarnings("ignore")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)

# Load the engineered matrices from Notebook 04
X_train = pd.read_csv("../data/telco_X_train_engineered.csv")
X_test = pd.read_csv("../data/telco_X_test_engineered.csv")
y_train = (pd.read_csv("../data/telco_y_train.csv")["Churn"] == "Yes").astype(int)
y_test = (pd.read_csv("../data/telco_y_test.csv")["Churn"] == "Yes").astype(int)

print(
    f"Loaded Engineered Feature Matrix: {X_train.shape[1]} features across {X_train.shape[0]} training samples."
)

## 1. Why Select Features?

More features do **not** necessarily make a better model!

1. **Occam's Razor & Generalization**: Simpler models with fewer moving parts generalize better to unseen production distributions.
2. **The Curse of Dimensionality**: As the number of features increases, the volume of the feature space grows exponentially, causing samples to become sparse and distant from one another.
3. **Reducing Collinearity & Variance**: Correlated features inflate coefficient variance in linear models and make feature interpretation noisy.
4. **Inference Latency & Production Cost**: Sourcing, transforming, and validating 20 features in real time is much cheaper and more resilient than supporting 200 features.

---

## 2. The Three Families of Feature Selection

```
Feature Selection Families
├── 1. Filter Methods (Model-agnostic; evaluates statistical properties of features)
│   ├── Variance Threshold (Removes near-constant columns)
│   ├── Pearson / Spearman Correlation Matrix (Linear collinearity)
│   └── Mutual Information (Captures non-linear dependencies)
├── 2. Wrapper Methods (Iteratively tests subsets using a machine learning model)
│   ├── Recursive Feature Elimination (RFE)
│   └── Sequential Forward / Backward Selection
└── 3. Embedded Methods (Selection performed naturally during model fitting)
    ├── L1 Regularization / Lasso (Pushes coefficients of irrelevant features to 0)
    ├── Tree Gini Importance (Measures total impurity decrease across splits)
    └── Permutation Importance (Measures score degradation upon shuffling)
```


### Family 1: Filter Methods

Filter methods rank each feature independently of any machine learning model. They are computationally fast ($O(N)$) and ideal for initial feature pruning.

#### A. Variance Threshold
Eliminates features whose variance falls below a threshold (e.g. features where 99.5% of values are identical).


In [ ]:
# TODO: Apply VarianceThreshold(threshold=0.01) to identify and remove near-constant features
# YOUR CODE HERE


#### B. Mutual Information (`mutual_info_classif`)
Unlike Pearson correlation (which only measures linear relationships), **Mutual Information (MI)** measures the reduction in uncertainty about target $y$ given knowledge of feature $X$:

$$I(X; y) = \sum_{x, y} p(x, y) \log \frac{p(x, y)}{p(x)p(y)}$$

An MI score of $0$ means $X$ and $y$ are completely independent; higher values indicate stronger non-linear predictive signal.


In [ ]:
# TODO: Calculate Mutual Information scores (mutual_info_classif) against Churn and plot top 15 features
# YOUR CODE HERE


### Family 2: Wrapper Methods (Recursive Feature Elimination)

**Wrapper methods** treat feature selection as a search problem.
**RFE (Recursive Feature Elimination)** fits a model on all features, ranks them by coefficient or feature importance, discards the weakest feature, and repeats until the desired number of features remains.


In [ ]:
# TODO: Apply Recursive Feature Elimination (RFE) with LogisticRegression to select top 12 features
# YOUR CODE HERE


### Family 3: Embedded Methods

Embedded methods select features during the model optimization process itself.

#### A. Random Forest Gini Importance vs Permutation Importance
- **Gini / MDI Importance**: Fast, but known to artificially favor high-cardinality continuous features.
- **Permutation Importance**: Measures how much the model's test/validation score drops when values of a single feature are randomly shuffled. If shuffling column $j$ crashes model performance, column $j$ is vital!


In [ ]:
# TODO: Fit a RandomForestClassifier baseline and compute Permutation Importance on test set
# Plot top 15 features by permutation importance
# YOUR CODE HERE


## 3. Selecting the Optimal Feature Subset

Combining Mutual Information and Permutation Importance insights, we select the top **12 most predictive, non-redundant features** for downstream modeling:
- `tenure`, `MonthlyCharges`, `TotalCharges_log`, `charge_tenure_ratio`, `total_services`
- `Contract_target_enc`, `Contract_One year`, `Contract_Two year`
- `InternetService_Fiber optic`, `PaymentMethod_Electronic check`, `PaperlessBilling_Yes`, `SeniorCitizen`


In [ ]:
# TODO: Select final feature subset based on selection findings and save to ../data/
# X_train_selected.to_csv('../data/telco_X_train_selected.csv', index=False)
# X_test_selected.to_csv('../data/telco_X_test_selected.csv', index=False)
# YOUR CODE HERE
